# 📓 NB-04 — Atividade Avaliativa: Agente de Patrulha Autônomo
## ESP004 — Sistemas Autônomos Robóticos | PPGESP/IFBA 2026.2
**Autores:** Prof. Dr. José Alberto Díaz Amado · Prof. Dr. João Erivando Soares Marques

---
## 📋 Instruções

Este notebook é o **template oficial** da atividade avaliativa.
Complete as células marcadas com `# TODO` e entregue o notebook commitado no GitHub.

### Rubrica (100 pontos)
| Etapa | Pontos |
|-------|--------|
| Repositório GitHub correto (estrutura, .gitignore, ≥5 commits) | 10 |
| Máquina de estados com 4 estados implementados | 20 |
| Robô percorre quadrado 2m×4 no Gazebo sem travar | 15 |
| Parada segura ao detectar obstáculo + retomada | 15 |
| Bag commitado com ≥500 amostras de /odom | 10 |
| Notebook com trajetória + métricas commitado | 15 |
| Modelo treinado no Colab (≥80% acc) commitado | 10 |
| Nó com PyTorch operacional no ROS 2 | 5 |

---
## 📦 Identificação do Aluno

In [1]:
# ═══════════════════════════════════════════════════════════
# IDENTIFICAÇÃO — Preencha seus dados
# ═══════════════════════════════════════════════════════════

ALUNO_NOME   = "Jonas Magalhães Silva"
ALUNO_EMAIL  = "jonasmagalhaes.8@gmail.com"
GITHUB_USER  = "jonasmagalhaes-8"
REPO_URL     = "https://github.com/jonasmagalhaes-8/ESP004_NB04_Atividade_Avaliativa"
DATA_ENTREGA = "2026-11-12"

print('=' * 55)
print('  ATIVIDADE AVALIATIVA — ESP004')
print('  Agente de Patrulha Autônomo')
print('=' * 55)
print(f'Aluno:    {ALUNO_NOME}')
print(f'Email:    {ALUNO_EMAIL}')
print(f'GitHub:   {GITHUB_USER}')
print(f'Repo:     {REPO_URL}')
print(f'Entrega:  {DATA_ENTREGA}')


  ATIVIDADE AVALIATIVA — ESP004
  Agente de Patrulha Autônomo
Aluno:    Jonas Magalhães Silva
Email:    jonasmagalhaes.8@gmail.com
GitHub:   jonasmagalhaes
Repo:     https://github.com/jonasmagalhaes/esp004-patrulha
Entrega:  2026-11-12


---
## Etapa 1 — Verificar Repositório GitHub (10 pts)

In [2]:
# ═══════════════════════════════════════════════════════════
# ETAPA 1 — Verificar repositório e estrutura
# ═══════════════════════════════════════════════════════════
import subprocess, os

# Verificar se estamos dentro de um repositório git
r = subprocess.run(['git', 'rev-parse', '--show-toplevel'],
                   capture_output=True, text=True)
if r.returncode == 0:
    repo_root = r.stdout.strip()
    print(f'✅ Repositório git encontrado: {repo_root}')
else:
    print('🔴 NÃO está dentro de um repositório git!')
    print('   Execute: git init && git remote add origin <URL>')

# Verificar commits
r2 = subprocess.run(['git', 'log', '--oneline'],
                    capture_output=True, text=True)
commits = [l for l in r2.stdout.strip().split('\n') if l]
n_commits = len(commits)
print(f'\nCommits: {n_commits}')
for c in commits[:8]:
    print(f'  {c}')

if n_commits >= 5:
    print(f'\n🟢 {n_commits} commits — Requisito cumprido (≥5)')
else:
    print(f'\n🔴 Apenas {n_commits} commits. Faça mais commits conforme avança.')

# Verificar .gitignore
gi = os.path.join(repo_root if r.returncode==0 else '.', '.gitignore')
if os.path.exists(gi):
    print('🟢 .gitignore encontrado')
else:
    print('🔴 .gitignore não encontrado — crie-o!')

✅ Repositório git encontrado: /workspace

Commits: 12
  02ccf3f feat(nb04): executa PatrolNode 4 lados completos com metricas finais
  386597c feat(nb03): executa treino da ObstacleNet e controle interativo
  e643239 feat(nb02): executa exemplos e solucao do QuadradoNode com sucesso
  bac0f68 feat(nb01): executa e valida ambiente ROS 2 Jazzy e Python sem erros
  4956b22 feat(atividade): solução final PatrolNode [2026-11-12 19:28]
  ac16df0 feat(analise): adiciona analise grafica da patrulha e metricas
  60f995a feat(telemetria): adiciona rosbag com 599 amostras de /odom
  475c9ee feat(notebooks): adiciona painel interativo e pipeline de treino no NB03

🟢 12 commits — Requisito cumprido (≥5)
🟢 .gitignore encontrado


---
## Etapa 2 — Máquina de Estados do Agente de Patrulha (20 pts)

In [3]:
# ═══════════════════════════════════════════════════════════
# ETAPA 2 — Implementar PatrolNode com máquina de estados
# Complete as partes marcadas com TODO
# ═══════════════════════════════════════════════════════════
import rclpy, threading, time, math
from rclpy.node import Node
from geometry_msgs.msg import Twist
from std_msgs.msg import Bool


class PatrolNode(Node):
    """
    Agente de patrulha com máquina de estados.
    Estados: MOVING_FORWARD → TURNING → MOVING_FORWARD → ... → DONE
    Interrupção: qualquer estado → AVOIDING (quando obstáculo)
    """

    # Definição dos estados
    MOVING_FORWARD = 'MOVING_FORWARD'
    TURNING        = 'TURNING'
    AVOIDING       = 'AVOIDING'
    DONE           = 'DONE'

    # Constantes de velocidade
    VEL_LINEAR  = 0.3   # m/s
    VEL_ANGULAR = 0.785 # rad/s  ≈ 90°/2s
    DIST_LADO   = 2.0   # metros por lado do quadrado
    N_LADOS     = 4     # lados do quadrado

    def __init__(self):
        super().__init__('patrol')

        # Criar publisher em /cmd_vel (tipo Twist)
        self._pub_vel = self.create_publisher(Twist, '/cmd_vel', 10)

        # Criar subscription em /obstaculo (tipo Bool)
        self._sub_obs = self.create_subscription(
            Bool, '/obstaculo', self._obstaculo_cb, 10)

        # Estado inicial
        self._estado   = self.MOVING_FORWARD
        self._lados    = 0
        self._t_estado = time.time()  # timestamp de entrada no estado
        self._dist_acc = 0.0          # distância acumulada no estado atual
        self._obstaculo = False
        self._estado_anterior = None  # para retomar após AVOIDING

        # Métricas para avaliação
        self.n_paradas = 0
        self.t_inicio  = time.time()
        self.historico = []  # (t, estado)

        # Timer de controle: 10 Hz
        self._timer = self.create_timer(0.1, self._step)
        self.get_logger().info('PatrolNode iniciado!')

    def _obstaculo_cb(self, msg: Bool):
        """Callback do tópico /obstaculo."""
        if msg.data and self._estado not in [self.AVOIDING, self.DONE]:
            self._estado_anterior = self._estado
            self._estado = self.AVOIDING
            self._t_estado = time.time()
            self.n_paradas += 1
            self.get_logger().warn('⚠️  Obstáculo detectado! → AVOIDING')
        elif not msg.data and self._estado == self.AVOIDING:
            # Obstáculo removido: retomar
            self._estado = self._estado_anterior or self.MOVING_FORWARD
            self._t_estado = time.time()
            self.get_logger().info('✅ Caminho livre. Retomando patrulha.')
        self._obstaculo = msg.data

    def _step(self):
        """Lógica da máquina de estados — chamada a 10 Hz."""
        msg = Twist()  # velocidade zero por padrão
        dt  = time.time() - self._t_estado
        self.historico.append((time.time() - self.t_inicio, self._estado))

        if self._estado == self.MOVING_FORWARD:
            dist_percorrida = self.VEL_LINEAR * dt  # d = v * t
            msg.linear.x = self.VEL_LINEAR
            if dist_percorrida >= self.DIST_LADO:
                self._estado = self.TURNING
                self._t_estado = time.time()
                self.get_logger().info(f'Lado percorrido ({self._lados + 1}/4). → TURNING')

        elif self._estado == self.TURNING:
            angulo_girado = self.VEL_ANGULAR * dt  # rad
            msg.angular.z = self.VEL_ANGULAR
            if angulo_girado >= math.pi / 2:  # 90°
                self._lados += 1
                if self._lados >= self.N_LADOS:
                    self._estado = self.DONE
                    self.get_logger().info('🏁 Patrulha completa! → DONE')
                else:
                    self._estado = self.MOVING_FORWARD
                self._t_estado = time.time()

        elif self._estado == self.AVOIDING:
            # Manter parado enquanto há obstáculo
            pass  # msg já é zero

        elif self._estado == self.DONE:
            pass  # parado, missão concluída

        self._pub_vel.publish(msg)


# ── Executar por 35s para completar o ciclo e mostrar resultados ──
if rclpy.ok(): rclpy.shutdown()
rclpy.init()

patrol = PatrolNode()
stop   = threading.Event()

t = threading.Thread(
    target=lambda e: [rclpy.spin_once(patrol, timeout_sec=0.05)
                      for _ in iter(lambda: e.is_set() or not rclpy.ok(), True)],
    args=(stop,), daemon=True)
t.start()

print('Patrulha em execução...')
print('Simulando obstáculos e monitorando máquina de estados...')

obs_pub = rclpy.create_node('obs_sim')
pub_obs = obs_pub.create_publisher(Bool, '/obstaculo', 10)

t0 = time.time()
while time.time() - t0 < 45.0 and patrol._estado != patrol.DONE:
    elapsed = time.time() - t0
    # Simular obstáculo entre 5s e 8s
    obs_msg = Bool()
    obs_msg.data = (5.0 < elapsed < 8.0)
    pub_obs.publish(obs_msg)

    if int(elapsed) % 2 == 0 and elapsed % 2 < 0.15:
        print(f't={elapsed:.1f}s | Estado: {patrol._estado} | '
              f'Lados: {patrol._lados}/4 | Paradas: {patrol.n_paradas}')
    time.sleep(0.1)

stop.set()
t.join(timeout=2)
patrol.destroy_node()
obs_pub.destroy_node()
rclpy.shutdown()

duracao = time.time() - patrol.t_inicio
print("\n✅ Missão encerrada!")
print(f'   Lados completados: {patrol._lados}/{patrol.N_LADOS}')
print(f'   Paradas por obstáculo: {patrol.n_paradas}')
print(f'   Duração total: {duracao:.1f}s')


[INFO] [1794512355.228051732] [patrol]: PatrolNode iniciado!


Patrulha em execução...
Simulando obstáculos e monitorando máquina de estados...
t=0.0s | Estado: MOVING_FORWARD | Lados: 0/4 | Paradas: 0
t=0.1s | Estado: MOVING_FORWARD | Lados: 0/4 | Paradas: 0


t=2.0s | Estado: MOVING_FORWARD | Lados: 0/4 | Paradas: 0
t=2.1s | Estado: MOVING_FORWARD | Lados: 0/4 | Paradas: 0


t=4.0s | Estado: MOVING_FORWARD | Lados: 0/4 | Paradas: 0
t=4.1s | Estado: MOVING_FORWARD | Lados: 0/4 | Paradas: 0


[WARN] [1794512360.268734681] [patrol]: ⚠️  Obstáculo detectado! → AVOIDING


t=6.0s | Estado: AVOIDING | Lados: 0/4 | Paradas: 1
t=6.1s | Estado: AVOIDING | Lados: 0/4 | Paradas: 1


... (21 outputs left out)

[INFO] [1794512396.280921175] [patrol]: Lado percorrido (4/4). → TURNING


t=42.0s | Estado: TURNING | Lados: 3/4 | Paradas: 1
t=42.1s | Estado: TURNING | Lados: 3/4 | Paradas: 1



✅ Missão encerrada!
   Lados completados: 4/4
   Paradas por obstáculo: 1
   Duração total: 43.3s


[INFO] [1794512398.380342668] [patrol]: 🏁 Patrulha completa! → DONE


---
## Etapa 3 — Integração com ObstacleNet (PyTorch + ROS 2) (15 pts)
Nesta etapa, o nó ROS 2 carrega o modelo neural treinado ( e )
e realiza a inferência em tempo real, publicando no tópico  para interromper a patrulha de forma inteligente.


In [4]:
# ═══════════════════════════════════════════════════════════
# ETAPA 3 — Nó com PyTorch Operacional no ROS 2
# Inferência da ObstacleNet integrada ao ecossistema ROS 2
# ═══════════════════════════════════════════════════════════
import json, os, torch, rclpy, threading, time
import torch.nn as nn
import numpy as np
from rclpy.node import Node
from std_msgs.msg import Bool, Float32
from geometry_msgs.msg import Twist

# 1. Definição da arquitetura ObstacleNet
class ObstacleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),  nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# 2. Carregar scaler e modelo treinado
scaler_path = "models/scaler_config.json"
model_path  = "models/obstacle_net_v1.pt"

with open(scaler_path) as f:
    scaler_cfg = json.load(f)
mean = np.array(scaler_cfg["mean"], dtype=np.float32)
std  = np.array(scaler_cfg["std"], dtype=np.float32)

ia_model = ObstacleNet()
ia_model.load_state_dict(torch.load(model_path, map_location="cpu"))
ia_model.eval()
print("✅ Modelo ObstacleNet carregado com sucesso!")

# 3. Nó ROS 2 de inferência de obstáculos com IA
class ObstacleDetectorNode(Node):
    def __init__(self):
        super().__init__("obstacle_detector_ia")
        self.pub_obs   = self.create_publisher(Bool, "/obstaculo", 10)
        self.pub_prob  = self.create_publisher(Float32, "/obstaculo/prob", 10)
        self.sub_vel   = self.create_subscription(Twist, "/cmd_vel", self._vel_cb, 10)
        self.timer     = self.create_timer(0.2, self._inferir)  # 5 Hz
        self.cur_vel   = Twist()
        self.inferences = 0
        self.get_logger().info("ObstacleDetectorNode com PyTorch operacional!")

    def _vel_cb(self, msg: Twist):
        self.cur_vel = msg

    def _inferir(self):
        # Simular leitura de sensor frontal com aproximação realista
        t_curr = time.time()
        dist = 0.35 if (5.0 < (t_curr % 15.0) < 8.0) else 1.8
        
        features = np.array([
            self.cur_vel.linear.x,
            self.cur_vel.linear.y,
            dist,
            self.cur_vel.angular.z
        ], dtype=np.float32)
        
        feat_norm = (features - mean) / (std + 1e-8)
        feat_t    = torch.FloatTensor(feat_norm).unsqueeze(0)
        
        with torch.no_grad():
            prob = ia_model(feat_t).item()
            
        tem_obstaculo = (prob > 0.5)
        self.pub_obs.publish(Bool(data=tem_obstaculo))
        self.pub_prob.publish(Float32(data=float(prob)))
        self.inferences += 1

# Teste de validação do nó ROS 2 com PyTorch
if rclpy.ok(): rclpy.shutdown()
rclpy.init()
detector = ObstacleDetectorNode()

for _ in range(5):
    rclpy.spin_once(detector, timeout_sec=0.25)

print(f"🟢 Nó PyTorch operacional no ROS 2: {detector.inferences} inferências executadas")
detector.destroy_node()
rclpy.shutdown()


✅ Modelo ObstacleNet carregado com sucesso!


[INFO] [1794512399.863011435] [obstacle_detector_ia]: ObstacleDetectorNode com PyTorch operacional!


🟢 Nó PyTorch operacional no ROS 2: 5 inferências executadas


---
## Etapa 4 — Análise da Sessão (15 pts)

In [5]:
import os
# ═══════════════════════════════════════════════════════════
# ETAPA 4 — Analisar histórico da sessão e plotar métricas
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if not patrol.historico:
    print('🔴 Execute a Etapa 2 primeiro.')
else:
    tempos  = [h[0] for h in patrol.historico]
    estados = [h[1] for h in patrol.historico]

    # Mapear estados para números
    estado_map = {
        'MOVING_FORWARD': 1,
        'TURNING':        2,
        'AVOIDING':       3,
        'DONE':           4,
    }
    nums = [estado_map.get(e, 0) for e in estados]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'Análise da Patrulha — {ALUNO_NOME}',
                 fontsize=12, fontweight='bold')

    # Gráfico 1: Linha do tempo dos estados
    cores = {1: 'steelblue', 2: 'darkorange', 3: 'crimson', 4: 'green'}
    for i in range(len(tempos)-1):
        axes[0].fill_between([tempos[i], tempos[i+1]],
                              [0, 0], [nums[i], nums[i]],
                              color=cores.get(nums[i], 'gray'), alpha=0.7)
    axes[0].set(yticks=[1,2,3,4],
                yticklabels=['MOVING\nFORWARD','TURNING','AVOIDING','DONE'],
                xlabel='Tempo (s)', title='Linha do Tempo dos Estados')
    axes[0].grid(True, axis='x', alpha=0.3)

    # Legenda
    patches = [mpatches.Patch(color=c, label=n)
               for n, c in zip(['MOVING','TURNING','AVOIDING','DONE'],
                               ['steelblue','darkorange','crimson','green'])]
    axes[0].legend(handles=patches, loc='upper right', fontsize=8)

    # Gráfico 2: Distribuição de tempo por estado
    from collections import Counter
    contagem = Counter(estados)
    nomes = list(contagem.keys())
    vals  = list(contagem.values())
    cmap  = ['steelblue' if n=='MOVING_FORWARD' else
             'darkorange' if n=='TURNING' else
             'crimson' if n=='AVOIDING' else 'green' for n in nomes]
    axes[1].bar(nomes, [v/10 for v in vals], color=cmap, edgecolor='white')
    axes[1].set(xlabel='Estado', ylabel='Tempo (s)',
                title='Tempo Total por Estado')
    axes[1].tick_params(axis='x', labelsize=8)
    axes[1].grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    os.makedirs('resultados', exist_ok=True)
    plt.savefig('resultados/analise_patrulha.png', dpi=120)
    plt.show()

    # Métricas textuais
    print('\n=== MÉTRICAS DA MISSÃO ===')
    print(f'Duração total:          {max(tempos):.1f} s')
    print(f'Lados completos:        {patrol._lados}/4')
    print(f'Paradas por obstáculo:  {patrol.n_paradas}')
    t_mov = contagem.get('MOVING_FORWARD', 0) / 10
    t_tur = contagem.get('TURNING', 0) / 10
    t_avi = contagem.get('AVOIDING', 0) / 10
    print(f'Tempo em movimento:     {t_mov:.1f} s ({t_mov/max(tempos)*100:.0f}%)')
    print(f'Tempo virando:          {t_tur:.1f} s ({t_tur/max(tempos)*100:.0f}%)')
    print(f'Tempo evitando:         {t_avi:.1f} s ({t_avi/max(tempos)*100:.0f}%)')
    print('\n📊 Gráfico salvo em resultados/analise_patrulha.png')


<Figure size 1300x400 with 2 Axes>


=== MÉTRICAS DA MISSÃO ===
Duração total:          43.3 s
Lados completos:        4/4
Paradas por obstáculo:  1
Tempo em movimento:     31.9 s (74%)
Tempo virando:          8.3 s (19%)
Tempo evitando:         3.0 s (7%)

📊 Gráfico salvo em resultados/analise_patrulha.png


---
## Etapa 5 — Commit Final (para nota)

In [6]:
# ═══════════════════════════════════════════════════════════
# COMMIT FINAL — Execute após completar todas as etapas
# ═══════════════════════════════════════════════════════════
import subprocess, datetime

timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

cmds = [
    'git add .',
    f'git commit -m "feat(atividade): solução final PatrolNode [{timestamp}]"'
]

for cmd in cmds:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    saida = r.stdout or r.stderr
    if saida.strip():
        print(saida[:300])
    print()

print('='*55)
print('ENTREGA CONCLUÍDA!')
print(f'Aluno:  {ALUNO_NOME}')
print(f'Repo:   {REPO_URL}')
print(f'Data:   {timestamp}')
print('='*55)
print("\n📧 Envie o link do repositório no SUAP.")


$ git add .

$ git commit -m "feat(atividade): solução final PatrolNode [2026-11-12 19:40]"
[main b2fe17a] feat(atividade): solução final PatrolNode [2026-11-12 19:40]
 2 files changed, 216 insertions(+), 112 deletions(-)


ENTREGA CONCLUÍDA!
Aluno:  Jonas Magalhães Silva
Repo:   https://github.com/jonasmagalhaes/esp004-patrulha
Data:   2026-11-12 19:40

📧 Envie o link do repositório no SUAP.
